# V6 on Kaggle GPU (T4 / P100)

End-to-end training + rolling-origin CV for the V6 forecaster.
Attach the dataset that `scripts/push_to_kaggle.sh` uploaded
(expected files: `abt_v6_cached.parquet`, `v6_feature_manifest.json`,
`src_and_scripts.zip`).

## 1. Environment sanity check

In [ ]:
!nvidia-smi | head -5
import sys, platform
print('Python', platform.python_version(), sys.executable)

## 2. Install dependencies & unpack source

In [ ]:
!pip install -q lightgbm==4.6.0 joblib pandas pyarrow matplotlib scikit-learn optuna
import os, shutil, zipfile
INPUT_DIR = '/kaggle/input'
DATASET = [d for d in os.listdir(INPUT_DIR) if 'bpm' in d.lower() or 'v6' in d.lower()][0]
print('Using dataset:', DATASET)
SRC_ZIP = f'{INPUT_DIR}/{DATASET}/src_and_scripts.zip'
with zipfile.ZipFile(SRC_ZIP) as z:
    z.extractall('/kaggle/working/repo')
import sys
sys.path.insert(0, '/kaggle/working/repo')
shutil.copy(f'{INPUT_DIR}/{DATASET}/abt_v6_cached.parquet', '/kaggle/working/repo/output/abt_v6_cached.parquet') if os.path.isdir('/kaggle/working/repo/output') else None

## 3. Train V6 (fixed split, pinball-q60)

In [ ]:
import os
os.chdir('/kaggle/working/repo')
!mkdir -p output && cp /kaggle/input/$(ls /kaggle/input | head -1)/abt_v6_cached.parquet output/ 2>/dev/null || true
!python -m scripts.train_v6 --num-boost-round 1500

## 4. Rolling-origin cross-validation

In [ ]:
!python -m scripts.rolling_origin_cv --abt output/abt_v6_cached.parquet \
    --target target_qty_imputed --reg-objective pinball --alpha 0.6 \
    --n-origins 8 --num-boost-round 800 \
    --output-json output/v6_rolling_cv_gpu.json --output-md output/v6_rolling_cv_gpu.md

## 5. Optional: Optuna 500-trial retune
Only run if the core pipeline does not hit ≈0.47 test WAPE.

In [ ]:
# import optuna  # uncomment to enable full retune
# run the Optuna objective across num_leaves, learning_rate, feature_fraction, alpha
# save best_params to output/v6_optuna_best_params.json
print('Skipped by default.')

## 6. Export artefacts to `/kaggle/working` for pull-back

In [ ]:
import shutil
for f in ['output/model_v6.joblib', 'output/v6_rolling_cv_gpu.json',
          'output/v6_rolling_cv_gpu.md', 'output/feature_importance_v6.csv',
          'output/v6_metrics.csv', 'output/v6_vs_v5.md']:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/')
print('Artefacts copied:', os.listdir('/kaggle/working'))